# 1. Feature Engineering & Chuẩn hóa Data cho DataFlow26
Notebook này nhằm mục đích giải quyết **Data Leakage** (rò rỉ dữ liệu) từ file `daily_sales_data_final.csv` bằng cách thêm các Lag features và Rolling window. Đồng thời kết hợp với Web Traffic và thiết lập Calendar features để tạo data huấn luyện chuẩn.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

ModuleNotFoundError: No module named 'pandas'

### 2. Đọc dữ liệu daily_sales gộp

In [ ]:
data_path = 'data'
sales_df = pd.read_csv(os.path.join(data_path, 'daily_sales_data_final.csv'))
sales_df['date'] = pd.to_datetime(sales_df['date'])
sales_df = sales_df.sort_values('date').reset_index(drop=True)

print('Shape:', sales_df.shape)
display(sales_df.head(3))

### 3. Tạo Features từ Thời gian (Calendar Features)
Doanh thu bị ảnh hưởng nhiều bởi các chu kỳ: Thứ mấy trong tuần, tháng lãnh lương hay cuối tuần. Tree models rất cần những thuộc tính này.

In [ ]:
sales_df['year'] = sales_df['date'].dt.year
sales_df['month'] = sales_df['date'].dt.month
sales_df['day'] = sales_df['date'].dt.day
sales_df['dayofweek'] = sales_df['date'].dt.dayofweek
sales_df['is_weekend'] = sales_df['dayofweek'].isin([5, 6]).astype(int)
sales_df['quarter'] = sales_df['date'].dt.quarter


### 4. Xử lý Leakage bằng Lags & Tín hiệu Lịch sử
**QUAN TRỌNG:** Ở ngày dự báo tương lai, ta KHÔNG biết `quantity`, `order_id` hay `gross_profit`. Nếu train thẳng sẽ bị Leakage cực nặng.
--> Ta phải tạo LAG (trễ) để dịch những cột này về 1 ngày, 7 ngày trước. Khi đó input của mô hình sẽ là: "Lượng order hôm qua, doanh thu hôm qua" thay vì "doanh thu hôm nay".

In [ ]:
lag_features = [
    'actual_revenue', 'gross_profit', 'order_id', 'quantity', 'customer_id', 'is_returned',
    'revenue_category_Casual', 'revenue_color_black', 'revenue_category_Streetwear'
]
lag_features = [col for col in lag_features if col in sales_df.columns]

# Sinh Lag 1 ngày và Lag 7 ngày (1 tuần trước vì có tính chu kỳ)
for col in lag_features:
    sales_df[f'{col}_lag1'] = sales_df[col].shift(1)
    sales_df[f'{col}_lag7'] = sales_df[col].shift(7)

### 5. Rolling Window (Đường trung bình động)
Tính đường trung bình doanh thu 7 ngày để mô hình tự học Trend (Xu hướng).

In [ ]:
# Tính trên lag1 để không dính Revenue hiện tại
sales_df['revenue_roll_mean_7_lag1'] = sales_df['actual_revenue'].shift(1).rolling(window=7).mean()
sales_df['revenue_roll_std_7_lag1'] = sales_df['actual_revenue'].shift(1).rolling(window=7).std()

# Đánh dấu các cột Leakage cần bị loại bỏ trước khi train model
leakage_cols = [c for c in sales_df.columns if 'revenue_category_' in c or 'revenue_segment' in c or 'revenue_color_' in c]
leakage_cols += ['gross_profit', 'order_id', 'quantity', 'customer_id', 'is_returned']

print("Các cột sau sẽ không được dùng để train trực tiếp:", leakage_cols)

### 6. Bổ sung Traffic (Web Traffic Fact) làm mồi nhử cho Model
Nối Traffic (sessions, visitors) lag_1 làm feature để dự báo Doanh thu tương lai.

In [ ]:
try:
    traffic_df = pd.read_csv(os.path.join(data_path, 'Fact_WebTraffic.csv'))
    # Quy chuẩn cột date
    date_col = next((c for c in traffic_df.columns if c.lower() == 'date'), None)
    if date_col:
        traffic_df['date'] = pd.to_datetime(traffic_df[date_col])
        
        # Tạo lag cho sessions và page_views
        traffic_df['sessions_lag1'] = traffic_df['sessions'].shift(1) if 'sessions' in traffic_df.columns else None
        
        # Merge
        sales_df = sales_df.merge(traffic_df[['date', 'sessions_lag1']], on='date', how='left')
        print("Đã gộp thành công Traffic Lag!")
except Exception as e:
    print("Bỏ qua gộp Web Traffic do lỗi:", e)

### 7. Xuất Tập Dữ liệu Train Ready
Loại bỏ các dòng trống do quá trình Shift (Lag) sinh ra. Bỏ đi các thuộc tính bị Data Leakage. Kết quả cuối là 1 bảng features sạch để ném ngay vào LightGBM / XGBoost.

In [ ]:
final_df = sales_df.dropna()
final_train_cols = [c for c in final_df.columns if c not in leakage_cols]
train_ready_df = final_df[final_train_cols]

print("Shape Final Training Data:", train_ready_df.shape)
train_ready_df.to_csv(os.path.join(data_path, 'train_ready_dataset.csv'), index=False)
print("Đã lưu 'train_ready_dataset.csv'!")